# Phase-2 Step 9: Statistical & Reproducibility Checks

**Objective:** Ensure results are statistically valid and reproducible.

**Tasks:**
1. Multiple runs with different seeds
2. Report mean ± std for all metrics
3. Statistical significance tests
4. Confidence intervals

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_splits import create_within_battery_split, normalize_features

print("✅ Setup complete")

## 1. Load Data

In [ ]:
# Load data
all_features = pd.read_csv('data/phase2/harmonized/all_features.csv')

# Split
splits = create_within_battery_split(all_features, train_ratio=0.7, val_ratio=0.15)
train_df, val_df, test_df = splits['train'], splits['val'], splits['test']

# Features
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in all_features.columns if c not in exclude_cols]

# Normalize
train_norm, val_norm, test_norm, _ = normalize_features(train_df, val_df, test_df, exclude_cols)

# Prepare data
X_train = np.vstack([train_norm[feature_cols].values, val_norm[feature_cols].values])
y_train = np.hstack([train_norm['soh'].values, val_norm['soh'].values])
X_test = test_norm[feature_cols].values
y_test = test_norm['soh'].values

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 2. Multiple Runs with Different Seeds

In [ ]:
N_RUNS = 10
SEEDS = list(range(42, 42 + N_RUNS))

print(f"Running {N_RUNS} experiments with seeds: {SEEDS}")

In [ ]:
def run_experiment(X_train, y_train, X_test, y_test, model_class, seed):
    """Run a single experiment with given seed."""
    np.random.seed(seed)
    
    if model_class == Ridge:
        model = Ridge(alpha=1.0)
    elif model_class == GradientBoostingRegressor:
        model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
    elif model_class == RandomForestRegressor:
        model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=seed)
    else:
        raise ValueError(f"Unknown model: {model_class}")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'r2': r2_score(y_test, y_pred),
        'seed': seed
    }

In [ ]:
# Run experiments for each model
models = {
    'Ridge': Ridge,
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor
}

all_results = {}

print("="*70)
print(f"RUNNING {N_RUNS} EXPERIMENTS PER MODEL")
print("="*70)

for model_name, model_class in models.items():
    print(f"\n{model_name}:")
    results = []
    
    for seed in SEEDS:
        result = run_experiment(X_train, y_train, X_test, y_test, model_class, seed)
        results.append(result)
    
    all_results[model_name] = results
    
    # Summary
    r2_values = [r['r2'] for r in results]
    mae_values = [r['mae'] for r in results]
    
    print(f"  R²:  {np.mean(r2_values):.4f} ± {np.std(r2_values):.4f}")
    print(f"  MAE: {np.mean(mae_values):.4f} ± {np.std(mae_values):.4f}")

## 3. Statistical Summary

In [ ]:
# Create summary table
summary_data = []

for model_name, results in all_results.items():
    r2_values = [r['r2'] for r in results]
    mae_values = [r['mae'] for r in results]
    rmse_values = [r['rmse'] for r in results]
    
    # 95% confidence interval
    r2_ci = stats.t.interval(0.95, len(r2_values)-1, 
                              loc=np.mean(r2_values), 
                              scale=stats.sem(r2_values))
    
    summary_data.append({
        'Model': model_name,
        'R² (mean)': np.mean(r2_values),
        'R² (std)': np.std(r2_values),
        'R² (95% CI)': f"[{r2_ci[0]:.4f}, {r2_ci[1]:.4f}]",
        'MAE (mean)': np.mean(mae_values),
        'MAE (std)': np.std(mae_values),
        'RMSE (mean)': np.mean(rmse_values),
        'RMSE (std)': np.std(rmse_values),
        'N runs': len(results)
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ['r2', 'mae', 'rmse']
titles = ['R² Distribution', 'MAE Distribution', 'RMSE Distribution']

for ax, metric, title in zip(axes, metrics, titles):
    data = []
    labels = []
    
    for model_name, results in all_results.items():
        values = [r[metric] for r in results]
        data.append(values)
        labels.append(model_name)
    
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    
    colors = ['steelblue', 'forestgreen', 'coral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(metric.upper())
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Metric Distributions ({N_RUNS} runs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/step9_metric_distributions.png', dpi=150)
plt.show()

print("✅ Saved: results/phase2/figures/step9_metric_distributions.png")

## 4. Statistical Significance Tests

In [ ]:
print("\n" + "="*70)
print("STATISTICAL SIGNIFICANCE TESTS (Paired t-test)")
print("="*70)

# Compare Ridge vs Gradient Boosting
ridge_r2 = [r['r2'] for r in all_results['Ridge']]
gb_r2 = [r['r2'] for r in all_results['Gradient Boosting']]
rf_r2 = [r['r2'] for r in all_results['Random Forest']]

# Ridge vs GB
t_stat, p_value = stats.ttest_rel(ridge_r2, gb_r2)
print(f"\nRidge vs Gradient Boosting:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant (p<0.05): {'Yes' if p_value < 0.05 else 'No'}")

# Ridge vs RF
t_stat, p_value = stats.ttest_rel(ridge_r2, rf_r2)
print(f"\nRidge vs Random Forest:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant (p<0.05): {'Yes' if p_value < 0.05 else 'No'}")

# GB vs RF
t_stat, p_value = stats.ttest_rel(gb_r2, rf_r2)
print(f"\nGradient Boosting vs Random Forest:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant (p<0.05): {'Yes' if p_value < 0.05 else 'No'}")

## 5. Reproducibility Check

In [ ]:
print("\n" + "="*70)
print("REPRODUCIBILITY CHECK")
print("="*70)

# Run same experiment twice with same seed
seed = 42
result1 = run_experiment(X_train, y_train, X_test, y_test, GradientBoostingRegressor, seed)
result2 = run_experiment(X_train, y_train, X_test, y_test, GradientBoostingRegressor, seed)

print(f"\nSame seed ({seed}) run twice:")
print(f"  Run 1 R²: {result1['r2']:.10f}")
print(f"  Run 2 R²: {result2['r2']:.10f}")
print(f"  Difference: {abs(result1['r2'] - result2['r2']):.2e}")

if abs(result1['r2'] - result2['r2']) < 1e-10:
    print("  ✅ Results are EXACTLY reproducible")
else:
    print("  ⚠️ Small numerical differences detected")

## 6. Save Results

In [ ]:
# Save summary
summary_df.to_csv('results/phase2/step9_statistical_summary.csv', index=False)
print("✅ Saved: results/phase2/step9_statistical_summary.csv")

# Save all run details
all_runs_data = []
for model_name, results in all_results.items():
    for r in results:
        all_runs_data.append({
            'model': model_name,
            'seed': r['seed'],
            'r2': r['r2'],
            'mae': r['mae'],
            'rmse': r['rmse']
        })

runs_df = pd.DataFrame(all_runs_data)
runs_df.to_csv('results/phase2/step9_all_runs.csv', index=False)
print("✅ Saved: results/phase2/step9_all_runs.csv")

In [ ]:
print("\n" + "="*70)
print("STEP 9: SUMMARY")
print("="*70)

print(f"\n📊 Experiments run: {N_RUNS} per model")
print(f"📊 Models tested: {len(models)}")

# Best model
best_model = summary_df.loc[summary_df['R² (mean)'].idxmax(), 'Model']
best_r2 = summary_df.loc[summary_df['R² (mean)'].idxmax(), 'R² (mean)']

print(f"\n🏆 Best model: {best_model} (R² = {best_r2:.4f})")

print("\n📋 Final Results (mean ± std):")
for _, row in summary_df.iterrows():
    print(f"   {row['Model']}: R² = {row['R² (mean)']:.4f} ± {row['R² (std)']:.4f}")

print("\n✅ Step 9 Complete - Proceed to Step 10: Final Report")